# PyTorch: Tensors, Autograd & Building a Network

CSCI 6379 · Topic 16. Assemble everything (neuron, layers, activations, backprop, gradient descent) into the standard PyTorch workflow: tensors, autograd, nn.Module, the training loop, and DataLoader.

## Tensors

A tensor is a multi-dimensional array like NumPy's, but it can run on a GPU and track gradients.

In [ ]:
import torch

a = torch.tensor([[1, 2], [3, 4]])
print("tensor:", a.tolist(), " shape:", tuple(a.shape))
print("matmul a@a:", (a @ a).tolist())          # [[7,10],[15,22]]
print("view:", torch.arange(6).view(2, 3).tolist())
print("zeros/ones/randn shapes:", tuple(torch.zeros(3,3).shape),
      tuple(torch.ones(2,2).shape), tuple(torch.randn(2,2).shape))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Autograd: backprop, automatic

Mark inputs with requires_grad=True, compute, call .backward(); PyTorch runs backpropagation and fills in .grad.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x + 2
z = 2 * y * y
z.mean().backward()
print("x.grad =", x.grad.tolist())     # 4/3 * y = [4.0, 5.333, 6.667]

## Define a model with nn.Module

nn.Linear(in, out) is the weighted-sum unit w.x + b for a whole layer of neurons.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)    # input -> hidden
        self.fc2 = nn.Linear(16, 1)    # hidden -> output
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleNN()
print(model)

## Put it together: train an MLP on the two moons

Two classes that curl around each other — no straight line separates them. The five-step training loop learns a curved boundary.

In [ ]:
import numpy as np
import torch.optim as optim

def make_moons(n=240, noise=0.15, seed=0):
    rng = np.random.default_rng(seed); k = n // 2
    t = np.linspace(0, np.pi, k)
    X = np.vstack([np.c_[np.cos(t), np.sin(t)],
                   np.c_[1 - np.cos(t), 0.5 - np.sin(t)]]) + rng.normal(0, noise, (2*k, 2))
    y = np.r_[np.zeros(k), np.ones(k)]
    return X.astype(np.float32), y.astype(np.float32)

X, y = make_moons()
Xt = torch.tensor(X); yt = torch.tensor(y).unsqueeze(1)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(400):
    optimizer.zero_grad()          # 1. clear old gradients
    outputs = model(Xt)            # 2. forward pass
    loss = criterion(outputs, yt)  # 3. measure error
    loss.backward()                # 4. backprop
    optimizer.step()               # 5. gradient-descent update
    losses.append(loss.item())

print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")

In [ ]:
# evaluate
model.eval()
with torch.no_grad():
    acc = ((torch.sigmoid(model(Xt)) >= 0.5).float() == yt).float().mean()
print("accuracy:", round(acc.item()*100, 1), "%")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses); ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)
gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
with torch.no_grad():
    P = torch.sigmoid(model(torch.tensor(np.c_[gx.ravel(), gy.ravel()], dtype=torch.float32))).numpy().reshape(gx.shape)
ax[1].contourf(gx, gy, P, levels=20, cmap="RdBu_r", alpha=.6)
ax[1].contour(gx, gy, P, levels=[0.5], colors="k")
ax[1].scatter(X[y==0,0], X[y==0,1], c="blue", s=12); ax[1].scatter(X[y==1,0], X[y==1,1], c="red", s=12)
ax[1].set_title("learned boundary"); plt.show()

## Mini-batches with DataLoader

For larger data, feed shuffled mini-batches instead of the whole set at once.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True)
for batch_X, batch_y in loader:
    print("batch shapes:", tuple(batch_X.shape), tuple(batch_y.shape))
    break